In [0]:
from pyspark.sql import functions as F

# ============================================================
# READ CLEAN SILVER
# ============================================================

df = spark.table("job_market.silver.jobs_clean")

total_jobs = df.count()

print("Clean Silver records:", total_jobs)

display(
    df.select(
        "job_id",
        "company",
        "job_title",
        "description"
    ).limit(10)
)

Clean Silver records: 2221


job_id company job_title description 5037833004 GuidePoint Security GPSU Military/Military Spouses - Commercial Fellowship <div class="content-intro"><p>GuidePoint Security provides trusted cybersecurity expertise, solutions and services that help organizations make better decisions and minimize risk. By taking a three-tiered, holistic approach for evaluating security posture and ecosystems, GuidePoint enables some of the nation’s top organizations, such as Fortune 500 companies and U.S. government agencies, to identify threats, optimize resources and integrate best-fit solutions that mitigate risk.</p></div><p>This application is for separating service members and military spouses looking to conduct a fellowship through DoD Skillbridge, Army Career Skills Program, Hiring Our Heroes or SpouseWorks. This application is for transitioning service members looking for<strong>&nbsp;remote commercial roles</strong> following their separation, or for military spouses conducting fellowships through the Career Accelerator: SpouseWorks.&nbsp;<span style="color: rgb(186, 55, 42);"><strong>These roles <u>do not</u> require a security clearance.</strong></span></p>
<p>As a fellow at GuidePoint Security, you will work closely with industry-leading subject matter experts to provide first-class cyber security services. Learners will complete in-house training programs, while working on projects to enhance their security knowledge and skills.</p>
<p><strong>Fellows will gain experience through:&nbsp;</strong></p>
<ul>
<li>Hands-On Technical Development</li>
<li>Real-World Security Response</li>
<li>Collaborative Project Work</li>
<li>Team and Position Shadowing</li>
<li>Partner-led Vendor Training and Certifications</li>
</ul>
<p><strong>Requirements:</strong></p>
<ul>
<li>You are separating from active-duty service within the next 18 months or are a Military Spouse who has been accepted into the SpouseWorks program.&nbsp;</li>
<li>You are eligible to conduct a fellowship through DoD Skillbridge, Army Career Skills Program, Hiring Our Heroes or SpouseWorks.</li>
<li>For technical roles, you will need at least a basic knowledge of networking, common security tools and practices, as well as a passion for information security.</li>
<li>Preference is given to those currently serving in IT or cyber positions.</li>
</ul>
<p>&nbsp;For more information on our GPSU program, please visit our&nbsp;<a href="https://www.guidepointsecurity.com/gpsu/">website</a>.</p>
<p>&nbsp;</p>
<p>&nbsp;</p>
<p><em>The below perks are for those that join our team full-time following separation.&nbsp;</em></p><div class="content-conclusion"><p><strong>We use Greenhouse Software as our applicant tracking system and Zoom Scheduler for HR screen request scheduling. At times, your email may block our communication with you. Please be sure to check your SPAM folder so that you don't miss updates on your application.</strong></p>
<p><strong><br>Why GuidePoint?<br><br></strong>GuidePoint Security is a rapidly growing, profitable, privately-held value added reseller that focuses exclusively on Information Security. Since its inception in 2011, GuidePoint has grown to over 1,300 employees, established strategic partnerships with leading security vendors, and serves as a trusted advisor to more than 6,200 customers.</p>
<p>Firmly-defined core values drive all aspects of the business, which have been paramount to the company’s success and establishment of an enjoyable workplace atmosphere. At GuidePoint, your colleagues are knowledgeable, skilled, and experienced and will seek to collaborate and provide mentorship and guidance at every opportunity. &nbsp;</p>
<p>This is a unique and rare opportunity to grow your career along with one of the fastest growing companies in the nation.<br><br><strong>Some added perks….</strong></p>
<ul>
<li>Remote workforce primarily (U.S. based only, some travel may be required for certain positions, working on-site may be required for Federal positions)

In [0]:
# ============================================================
# STANDARDIZED SKILL DICTIONARY
# ============================================================

skill_patterns = {
    "Python": r"\bpython\b",
    "SQL": r"\bsql\b",
    "PySpark": r"\bpyspark\b",
    "Apache Spark": r"\bapache spark\b|\bspark sql\b|\bspark\b",
    "Databricks": r"\bdatabricks\b",
    "AWS": r"\baws\b|\bamazon web services\b",
    "Amazon S3": r"\bs3\b|\bamazon s3\b",
    "Azure": r"\bazure\b|\bmicrosoft azure\b",
    "GCP": r"\bgcp\b|\bgoogle cloud platform\b",
    "BigQuery": r"\bbigquery\b",
    "Airflow": r"\bairflow\b|\bapache airflow\b",
    "Docker": r"\bdocker\b",
    "Kubernetes": r"\bkubernetes\b|\bk8s\b",
    "Kafka": r"\bkafka\b|\bapache kafka\b",
    "Snowflake": r"\bsnowflake\b",
    "PostgreSQL": r"\bpostgresql\b|\bpostgres\b",
    "MySQL": r"\bmysql\b",
    "MongoDB": r"\bmongodb\b|\bmongo db\b",
    "Java": r"\bjava\b",
    "Scala": r"\bscala\b",
    "C++": r"\bc\+\+\b",
    "C#": r"\bc#\b|\bc sharp\b",
    "JavaScript": r"\bjavascript\b|\bjs\b",
    "TypeScript": r"\btypescript\b|\bts\b",
    "Git": r"\bgit\b",
    "GitHub": r"\bgithub\b",
    "GitLab": r"\bgitlab\b",
    "REST API": r"\brest api\b|\brestful api\b|\brestful services\b",
    "API": r"\bapi\b|\bapis\b",
    "ETL": r"\betl\b|\bextract transform load\b",
    "ELT": r"\belt\b|\bextract load transform\b",
    "dbt": r"\bdbt\b",
    "Terraform": r"\bterraform\b",
    "Power BI": r"\bpower bi\b|\bpowerbi\b",
    "Tableau": r"\btableau\b",
    "Excel": r"\bexcel\b|\bmicrosoft excel\b",
    "Pandas": r"\bpandas\b",
    "NumPy": r"\bnumpy\b",
    "scikit-learn": r"\bscikit-learn\b|\bsklearn\b",
    "TensorFlow": r"\btensorflow\b",
    "PyTorch": r"\bpytorch\b",
    "Machine Learning": r"\bmachine learning\b",
    "Deep Learning": r"\bdeep learning\b",
    "NLP": r"\bnatural language processing\b|\bnlp\b",
    "LLM": r"\blarge language model\b|\blarge language models\b|\bllm\b|\bllms\b",
    "LangChain": r"\blangchain\b",
    "LangGraph": r"\blanggraph\b"
}

print("Number of skills configured:", len(skill_patterns))

Number of skills configured: 47


In [0]:
# ============================================================
# PREPARE DESCRIPTION
# ============================================================

skill_base = (
    df
    .select(
        "job_id",
        "company",
        "job_title",
        "source",
        "location_normalized",
        "is_remote",
        "description"
    )
    .withColumn(
        "description_clean",
        F.lower(
            F.coalesce(
                F.col("description"),
                F.lit("")
            )
        )
    )
)

display(
    skill_base.select(
        "job_id",
        "job_title",
        "description_clean"
    ).limit(5)
)

job_id job_title description_clean 5037833004 GPSU Military/Military Spouses - Commercial Fellowship <div class="content-intro"><p>guidepoint security provides trusted cybersecurity expertise, solutions and services that help organizations make better decisions and minimize risk. by taking a three-tiered, holistic approach for evaluating security posture and ecosystems, guidepoint enables some of the nation’s top organizations, such as fortune 500 companies and u.s. government agencies, to identify threats, optimize resources and integrate best-fit solutions that mitigate risk.</p></div><p>this application is for separating service members and military spouses looking to conduct a fellowship through dod skillbridge, army career skills program, hiring our heroes or spouseworks. this application is for transitioning service members looking for<strong>&nbsp;remote commercial roles</strong> following their separation, or for military spouses conducting fellowships through the career accelerator: spouseworks.&nbsp;<span style="color: rgb(186, 55, 42);"><strong>these roles <u>do not</u> require a security clearance.</strong></span></p>
<p>as a fellow at guidepoint security, you will work closely with industry-leading subject matter experts to provide first-class cyber security services. learners will complete in-house training programs, while working on projects to enhance their security knowledge and skills.</p>
<p><strong>fellows will gain experience through:&nbsp;</strong></p>
<ul>
<li>hands-on technical development</li>
<li>real-world security response</li>
<li>collaborative project work</li>
<li>team and position shadowing</li>
<li>partner-led vendor training and certifications</li>
</ul>
<p><strong>requirements:</strong></p>
<ul>
<li>you are separating from active-duty service within the next 18 months or are a military spouse who has been accepted into the spouseworks program.&nbsp;</li>
<li>you are eligible to conduct a fellowship through dod skillbridge, army career skills program, hiring our heroes or spouseworks.</li>
<li>for technical roles, you will need at least a basic knowledge of networking, common security tools and practices, as well as a passion for information security.</li>
<li>preference is given to those currently serving in it or cyber positions.</li>
</ul>
<p>&nbsp;for more information on our gpsu program, please visit our&nbsp;<a href="https://www.guidepointsecurity.com/gpsu/">website</a>.</p>
<p>&nbsp;</p>
<p>&nbsp;</p>
<p><em>the below perks are for those that join our team full-time following separation.&nbsp;</em></p><div class="content-conclusion"><p><strong>we use greenhouse software as our applicant tracking system and zoom scheduler for hr screen request scheduling. at times, your email may block our communication with you. please be sure to check your spam folder so that you don't miss updates on your application.</strong></p>
<p><strong><br>why guidepoint?<br><br></strong>guidepoint security is a rapidly growing, profitable, privately-held value added reseller that focuses exclusively on information security. since its inception in 2011, guidepoint has grown to over 1,300 employees, established strategic partnerships with leading security vendors, and serves as a trusted advisor to more than 6,200 customers.</p>
<p>firmly-defined core values drive all aspects of the business, which have been paramount to the company’s success and establishment of an enjoyable workplace atmosphere. at guidepoint, your colleagues are knowledgeable, skilled, and experienced and will seek to collaborate and provide mentorship and guidance at every opportunity. &nbsp;</p>
<p>this is a unique and rare opportunity to grow your career along with one of the fastest growing companies in the nation.<br><br><strong>some added perks….</strong></p>
<ul>
<li>remote workforce primarily (u.s. based only, some travel may be required for certain positions, working on-site may be required for federal positions)</li>
<li>group medica

In [0]:
# ============================================================
# EXTRACT JOB-SKILL RELATIONSHIPS
# ============================================================

skill_array = F.array(
    *[
        F.when(
            F.col("description_clean").rlike(pattern),
            F.lit(skill)
        )
        for skill, pattern in skill_patterns.items()
    ]
)

job_skill = (
    skill_base
    .withColumn(
        "skill_matches",
        F.expr(
            "filter(skill_matches_raw, x -> x IS NOT NULL)"
        )
    )
)

In [0]:
# ============================================================
# EXTRACT JOB-SKILL RELATIONSHIPS
# ============================================================

skill_array = F.array(
    *[
        F.when(
            F.col("description_clean").rlike(pattern),
            F.lit(skill)
        )
        for skill, pattern in skill_patterns.items()
    ]
)

job_skill = (
    skill_base
    .withColumn(
        "skill_matches_raw",
        skill_array
    )
    .withColumn(
        "skill_matches",
        F.expr(
            "filter(skill_matches_raw, x -> x IS NOT NULL)"
        )
    )
    .withColumn(
        "skill",
        F.explode("skill_matches")
    )
    .drop(
        "skill_matches_raw",
        "skill_matches",
        "description_clean"
    )
)

display(
    job_skill.limit(30)
)

job_id company job_title source location_normalized is_remote description skill 5390243 Cloudflare Senior Software Engineer, Durable Objects (DO) greenhouse in-office false <div class="content-intro"><h3>About Us</h3>
<p>At Cloudflare, we are on a mission to help build a better Internet. Today the company runs one of the world’s largest networks that powers millions of websites and other Internet properties for customers ranging from individual bloggers to SMBs to Fortune 500 companies. Cloudflare protects and accelerates any Internet application online without adding hardware, installing software, or changing a line of code. Internet properties powered by Cloudflare all have web traffic routed through its intelligent global network, which gets smarter with every request. As a result, they see significant improvement in performance and a decrease in spam and other attacks. Cloudflare was named to Entrepreneur Magazine’s Top Company Cultures list and ranked among the World’s Most Innovative Companies by Fast Company.</p>
<p>At Cloudflare, we’re not looking for people who wait for a polished roadmap; we’re looking for the builders who see the cracks in the Internet that everyone else has simply learned to live with. We value candidates who have the instinct to spot a "normalized" problem and the AI-native curiosity to create a solution using the latest tools. Our culture is built on iteration, leveraging AI to ship faster today to make it better tomorrow, while ensuring that every improvement, no matter how small, is shared across the team to lift everyone up. If you’re the type of person who values curiosity over bureaucracy, and that AI is a partner in solving tough problems to keep the Internet moving forward, you’ll fit right in.</p></div><h3><strong>Available Locations: Austin, TX</strong></h3>
<h3>About the Role</h3>
<p>Cloudflare’s Emerging Technologies &amp; Incubation team builds and launches bold new products on Cloudflare’s edge network. This role is on the Durable Objects team, which powers stateful serverless applications for Cloudflare customers. You’ll help evolve the runtime and the low-level routing and storage systems that support real-time chat, multiplayer games, AI agents, and other stateful workloads at scale.</p>
<h3>Responsibilities</h3>
<ul>
<li>Build and improve the Durable Objects runtime.</li>
<li>Design and develop the low-level routing and storage systems that power customer applications.</li>
<li>Improve reliability, scalability, performance, and feature depth.</li>
<li>Own code end-to-end from design through production.</li>
<li>Solve complex problems in production distributed systems.</li>
<li>Work across JavaScript, TypeScript, Rust, and C++.</li>
<li>Help drive the continued evolution of the system.</li>
</ul>
<h3>Desirable Skills, Knowledge &amp; Experience</h3>
<ul>
<li>5+ years of experience writing systems software.</li>
<li>Operational mindset with experience owning code in production.</li>
<li>Strong foundation in data structures, algorithms, and system design.</li>
<li>Deep expertise in at least one systems programming language such as C/C++ or Rust.</li>
<li>Experience designing and architecting large-scale distributed systems.</li>
<li>Familiarity with cloud platforms, especially serverless architectures.</li>
<li>Experience as a technical leader or team lead, driving engineering decisions.</li>
</ul>
<h3>Bonus Points</h3>
<ul>
<li>Experience working on database internals, especially SQLite.</li>
<li>Deep understanding of web technologies such as browsers, HTTP, JavaScript, and WebAssembly.</li>
<li>Background in low-latency, real-time environments such as game streaming, high-frequency trading, or payment systems.</li>
<li>Experience debugging, optimizing, and identifying failure modes in large-scale Linux-based distributed systems.</li>
</ul>
<h3 id="SeniorSystemsEngineerDurableObjects(DO)andD1-AbouttheDepartment"></h3><div class="content-conclusion"><h3>What Makes Cloudflare Special

In [0]:
# ============================================================
# JOB-SKILL VALIDATION
# ============================================================

job_skill_count = job_skill.count()

unique_job_skill_count = (
    job_skill
    .select(
        "job_id",
        "skill"
    )
    .distinct()
    .count()
)

print("Total job-skill rows :", job_skill_count)
print(
    "Unique job-skill pairs:",
    unique_job_skill_count
)

Total job-skill rows : 2961
Unique job-skill pairs: 2961


In [0]:
# ============================================================
# ENSURE ONE ROW PER JOB + SKILL
# ============================================================

job_skill = (
    job_skill
    .dropDuplicates(
        ["job_id", "skill"]
    )
)

print(
    "Final unique job-skill rows:",
    job_skill.count()
)

Final unique job-skill rows: 2961


In [0]:
# ============================================================
# SKILL DEMAND
# ============================================================

skill_demand = (
    job_skill
    .groupBy("skill")
    .agg(

        # Number of jobs requiring the skill
        F.countDistinct(
            "job_id"
        ).alias(
            "job_count"
        ),

        # Number of companies requiring it
        F.countDistinct(
            "company"
        ).alias(
            "company_count"
        ),

        # Number of locations
        F.countDistinct(
            "location_normalized"
        ).alias(
            "location_count"
        ),

        # Remote jobs
        F.countDistinct(
            F.when(
                F.col("is_remote") == True,
                F.col("job_id")
            )
        ).alias(
            "remote_job_count"
        ),

        # Number of different job titles
        F.countDistinct(
            "job_title"
        ).alias(
            "job_title_count"
        )
    )
)

In [0]:
# ============================================================
# SKILL DEMAND METRICS
# ============================================================

skill_demand = (
    skill_demand
    .withColumn(
        "demand_percentage",
        F.round(
            (
                F.col("job_count") /
                F.lit(total_jobs)
            ) * 100,
            2
        )
    )
    .withColumn(
        "remote_percentage",
        F.round(
            (
                F.col("remote_job_count") /
                F.col("job_count")
            ) * 100,
            2
        )
    )
)

In [0]:
# ============================================================
# SKILL DEMAND RANKING
# ============================================================
from pyspark.sql.window import Window

skill_rank_window = Window.orderBy(
    F.desc("job_count"),
    F.asc("skill")
)

skill_demand = (
    skill_demand
    .withColumn(
        "skill_demand_rank",
        F.dense_rank().over(
            skill_rank_window
        )
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# ============================================================
# SKILL DEMAND RANKING
# ============================================================

skill_rank_window = Window.orderBy(
    F.desc("job_count"),
    F.asc("skill")
)

skill_demand = (
    skill_demand
    .withColumn(
        "skill_demand_rank",
        F.dense_rank().over(
            skill_rank_window
        )
    )
)

display(skill_demand.limit(30))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


skill,job_count,company_count,location_count,remote_job_count,job_title_count,demand_percentage,remote_percentage,skill_demand_rank
API,493,9,132,85,441,22.2,17.24,1
Python,284,8,77,59,261,12.79,20.77,2
SQL,260,7,101,66,233,11.71,25.38,3
LLM,225,9,53,45,199,10.13,20.0,4
JavaScript,149,6,41,9,133,6.71,6.04,5
Java,133,6,57,31,95,5.99,23.31,6
TypeScript,133,7,38,7,108,5.99,5.26,7
AWS,123,7,42,42,114,5.54,34.15,8
Kubernetes,114,7,32,19,98,5.13,16.67,9
Machine Learning,104,8,38,26,91,4.68,25.0,10


In [0]:
# ============================================================
# FINAL SKILL DEMAND DATASET
# ============================================================

skill_demand = (
    skill_demand
    .select(
        "skill_demand_rank",
        "skill",
        "job_count",
        "demand_percentage",
        "company_count",
        "location_count",
        "job_title_count",
        "remote_job_count",
        "remote_percentage"
    )
    .orderBy(
        F.asc("skill_demand_rank")
    )
)

display(
    skill_demand.limit(30)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


skill_demand_rank,skill,job_count,demand_percentage,company_count,location_count,job_title_count,remote_job_count,remote_percentage
1,API,493,22.2,9,132,441,85,17.24
2,Python,284,12.79,8,77,261,59,20.77
3,SQL,260,11.71,7,101,233,66,25.38
4,LLM,225,10.13,9,53,199,45,20.0
5,JavaScript,149,6.71,6,41,133,9,6.04
6,Java,133,5.99,6,57,95,31,23.31
7,TypeScript,133,5.99,7,38,108,7,5.26
8,AWS,123,5.54,7,42,114,42,34.15
9,Kubernetes,114,5.13,7,32,98,19,16.67
10,Machine Learning,104,4.68,8,38,91,26,25.0


In [0]:
# ============================================================
# SAVE SKILL DEMAND GOLD TABLE
# ============================================================

skill_demand.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "job_market.gold.skill_demand"
    )

print("✓ Created:")
print("job_market.gold.skill_demand")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ Created:
job_market.gold.skill_demand


In [0]:
# ============================================================
# COMPANY × SKILL DEMAND
# ============================================================

company_skill_demand = (
    job_skill
    .groupBy(
        "company",
        "skill"
    )
    .agg(

        F.countDistinct(
            "job_id"
        ).alias(
            "job_count"
        ),

        F.countDistinct(
            "job_title"
        ).alias(
            "job_title_count"
        ),

        F.countDistinct(
            "location_normalized"
        ).alias(
            "location_count"
        ),

        F.countDistinct(
            F.when(
                F.col("is_remote") == True,
                F.col("job_id")
            )
        ).alias(
            "remote_job_count"
        )
    )
    .orderBy(
        F.desc("job_count")
    )
)

display(
    company_skill_demand.limit(30)
)

company,skill,job_count,job_title_count,location_count,remote_job_count
Stripe,API,223,199,109,29
Stripe,SQL,149,129,79,24
Cloudflare,API,149,142,6,0
Cloudflare,Python,105,101,6,1
Cloudflare,LLM,103,100,8,2
Stripe,Python,93,83,51,11
Cloudflare,JavaScript,91,89,5,1
Stripe,Java,71,56,40,6
Cloudflare,Tableau,65,64,3,0
Cloudflare,TypeScript,56,51,4,0


In [0]:
# ============================================================
# SAVE COMPANY × SKILL DEMAND
# ============================================================

company_skill_demand.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "job_market.gold.company_skill_demand"
    )

print("✓ Created:")
print("job_market.gold.company_skill_demand")

✓ Created:
job_market.gold.company_skill_demand


In [0]:
# ============================================================
# FINAL VALIDATION
# ============================================================

gold_skill = spark.table(
    "job_market.gold.skill_demand"
)

gold_company_skill = spark.table(
    "job_market.gold.company_skill_demand"
)

print("======================================")
print("SKILL DEMAND GOLD VALIDATION")
print("======================================")

print(
    "Clean Silver jobs:",
    total_jobs
)

print(
    "Unique job-skill pairs:",
    job_skill.select(
        "job_id",
        "skill"
    ).distinct().count()
)

print(
    "Skill demand rows:",
    gold_skill.count()
)

print(
    "Company × Skill rows:",
    gold_company_skill.count()
)

print("======================================")

print("✓ Skill Demand Gold layer completed.")

SKILL DEMAND GOLD VALIDATION
Clean Silver jobs: 2221
Unique job-skill pairs: 2961
Skill demand rows: 44
Company × Skill rows: 210
✓ Skill Demand Gold layer completed.
